In [1]:
!nvidia-smi
!pip install -q ultralytics supervision huggingface_hub pandas pyarrow
from google.colab import drive
drive.mount('/content/drive')

Sat Aug 22 07:59:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!find /content/drive/MyDrive -iname "*.MP4" 2>/dev/null | head

^C


In [2]:
import os
VID = "/content/drive/MyDrive/Visual Intelligence Hackathon/Intersection_Merged.MP4"  # ← paste from Cell 2
assert os.path.exists(VID), "WRONG PATH — fix before continuing"
print("ok", os.path.getsize(VID)/1e9, "GB")

ok 6.495679393 GB


In [3]:
%%writefile track.py
import argparse, inspect, time
from pathlib import Path
import cv2, numpy as np, pandas as pd
import supervision as sv
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

VISDRONE={0:"pedestrian",1:"people",2:"bicycle",3:"car",4:"van",5:"truck",
          6:"tricycle",7:"awning-tricycle",8:"bus",9:"motor"}
GROUP={"pedestrian":"pedestrian","people":"pedestrian","bicycle":"cyclist",
       "car":"car","van":"LGV","truck":"HGV","bus":"bus","motor":"motorcycle",
       "tricycle":"three-wheeler","awning-tricycle":"three-wheeler"}

def make_tracker(fps):
    sig=set(inspect.signature(sv.ByteTrack.__init__).parameters); kw={}
    if "frame_rate" in sig: kw["frame_rate"]=int(fps)
    for a,b in [("lost_track_buffer",int(fps*4)),("track_buffer",int(fps*4)),
                ("track_activation_threshold",0.20),("track_thresh",0.20),
                ("minimum_matching_threshold",0.85),("match_thresh",0.85)]:
        if a in sig and a not in kw: kw[a]=b
    print("[tracker]",kw,flush=True)
    return sv.ByteTrack(**kw)

_Q={"m":None}
def predict(model,imgs,device,half,tile,conf):
    if _Q["m"] is None:
        try:
            r=model.predict(imgs,device=device,imgsz=tile,conf=conf,verbose=False,
                            quantize=(16 if half else None))
            _Q["m"]="q"; return r
        except TypeError: _Q["m"]="h"
    if _Q["m"]=="q":
        return model.predict(imgs,device=device,imgsz=tile,conf=conf,verbose=False,
                             quantize=(16 if half else None))
    return model.predict(imgs,device=device,imgsz=tile,conf=conf,verbose=False,half=half)

def tiles(h,w,t,ov):
    s=max(int(t*(1-ov)),1)
    xs=sorted(set(list(range(0,max(w-t,1),s))+[max(w-t,0)]))
    ys=sorted(set(list(range(0,max(h-t,1),s))+[max(h-t,0)]))
    return [(x,y) for y in ys for x in xs]

def detect(model,fr,device,half,t,ov,batch,conf,iou):
    h,w=fr.shape[:2]; co=tiles(h,w,t,ov)
    cr=[fr[y:y+t,x:x+t] for x,y in co]; parts=[]
    for i in range(0,len(cr),batch):
        for r,(x,y) in zip(predict(model,cr[i:i+batch],device,half,t,conf),co[i:i+batch]):
            d=sv.Detections.from_ultralytics(r)
            if len(d):
                d.xyxy=d.xyxy+np.array([x,y,x,y],dtype=np.float32); parts.append(d)
    if not parts: return sv.Detections.empty()
    return sv.Detections.merge(parts).with_nms(threshold=iou,class_agnostic=True)

def frames(cap,s,e,st):
    if s: cap.set(cv2.CAP_PROP_POS_FRAMES,s)
    i=s
    while True:
        if e and i>=e: break
        if not cap.grab(): break
        if (i-s)%st==0:
            ok,f=cap.retrieve()
            if ok: yield i,f
        i+=1

def main():
    p=argparse.ArgumentParser()
    p.add_argument("--video",required=True); p.add_argument("--out-dir",default="/content/out")
    p.add_argument("--start-sec",type=float,default=0); p.add_argument("--max-seconds",type=float,default=0)
    p.add_argument("--stride",type=int,default=3); p.add_argument("--tile",type=int,default=1024)
    p.add_argument("--overlap",type=float,default=0.2); p.add_argument("--batch",type=int,default=12)
    p.add_argument("--conf",type=float,default=0.10); p.add_argument("--nms-iou",type=float,default=0.55)
    p.add_argument("--min-track-len",type=int,default=3); p.add_argument("--device",default="cuda")
    a=p.parse_args()
    out=Path(a.out_dir); out.mkdir(parents=True,exist_ok=True)
    half=a.device.startswith("cuda")
    model=YOLO(hf_hub_download("dronefreak/visdrone-yolov11s","best.pt"))
    cap=cv2.VideoCapture(a.video); assert cap.isOpened(), f"cannot open {a.video}"
    fps=cap.get(cv2.CAP_PROP_FPS) or 29.97
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W=int(cap.get(3)); H=int(cap.get(4))
    s=int(a.start_sec*fps); e=total if a.max_seconds<=0 else min(total,s+int(a.max_seconds*fps))
    n_samp=max((e-s)//a.stride,1)
    print(f"{W}x{H} @{fps:.2f} frames {s}->{e} sampled {n_samp}",flush=True)
    tracker=make_tracker(fps/a.stride)
    rows=[]; t0=time.time(); n=0
    for fi,fr in frames(cap,s,e,a.stride):
        det=tracker.update_with_detections(
            detect(model,fr,a.device,half,a.tile,a.overlap,a.batch,a.conf,a.nms_iou))
        for i in range(len(det)):
            if det.tracker_id is None or det.tracker_id[i] is None: continue
            x1,y1,x2,y2=det.xyxy[i]; cid=int(det.class_id[i]); cn=VISDRONE.get(cid,str(cid))
            rows.append(dict(frame_idx=fi,time_sec=fi/fps,track_id=int(det.tracker_id[i]),
                class_id=cid,class_name=cn,class_group=GROUP.get(cn,cn),
                conf=float(det.confidence[i]),x1=float(x1),y1=float(y1),
                x2=float(x2),y2=float(y2),cx=float((x1+x2)/2),cy_foot=float(y2),
                w=float(x2-x1),h=float(y2-y1),interpolated=False))
        n+=1
        if n%50==0:
            r=n/(time.time()-t0)
            print(f"{n}/{n_samp} {r:.2f} f/s ETA {(n_samp-n)/r/60:.1f} min",flush=True)
    cap.release()
    df=pd.DataFrame(rows)
    if df.empty: print("NO DETECTIONS"); return
    k=df.groupby("track_id").size(); df=df[df.track_id.isin(k[k>=a.min_track_len].index)].copy()
    v=(df.groupby(["track_id","class_group"])["conf"].sum().reset_index()
        .sort_values("conf",ascending=False).drop_duplicates("track_id")
        .set_index("track_id")["class_group"])
    df["class_group"]=df.track_id.map(v)
    df.to_parquet(out/"tracks.parquet",index=False)
    per=df.groupby("frame_idx").size(); L=df.groupby("track_id").size()
    print(f"\n=== RAW QC ===\nrows {len(df)} tracks {df.track_id.nunique()}")
    print(f"objects/frame mean {per.mean():.1f}")
    print(f"track len mean {L.mean():.1f} median {L.median():.0f}")
    print(f"wall {(time.time()-t0)/60:.1f} min")

if __name__=="__main__": main()

Writing track.py


In [4]:
%%writefile stitch.py
import sys, numpy as np, pandas as pd
def stitch(df,fps=29.97,max_gap_sec=2.0,gate_mult=3.0,min_gate=60.0):
    mg=int(max_gap_sec*fps)
    real=df[~df.interpolated] if "interpolated" in df else df
    info={}
    for tid,g in real.groupby("track_id"):
        g=g.sort_values("frame_idx"); n=min(4,len(g))
        def vel(s):
            if len(s)<2: return np.array([0.,0.])
            dt=s.frame_idx.iloc[-1]-s.frame_idx.iloc[0]
            if dt==0: return np.array([0.,0.])
            return np.array([(s.cx.iloc[-1]-s.cx.iloc[0])/dt,
                             (s.cy_foot.iloc[-1]-s.cy_foot.iloc[0])/dt])
        info[tid]=dict(f0=int(g.frame_idx.iloc[0]),f1=int(g.frame_idx.iloc[-1]),
            p0=np.array([g.cx.iloc[0],g.cy_foot.iloc[0]]),
            p1=np.array([g.cx.iloc[-1],g.cy_foot.iloc[-1]]),
            v0=vel(g.iloc[:n]),v1=vel(g.iloc[-n:]),
            size=float(np.sqrt(max(g.w.median()*g.h.median(),1))),cls=g.class_group.iloc[0])
    par={t:t for t in info}
    def find(x):
        while par[x]!=x: par[x]=par[par[x]]; x=par[x]
        return x
    byf0=sorted(info,key=lambda t:info[t]["f0"]); used=set(); pr=0
    for a in sorted(info,key=lambda t:info[t]["f1"]):
        A=info[a]; best=None; bd=1e18
        for b in byf0:
            if b==a or b in used: continue
            B=info[b]; gap=B["f0"]-A["f1"]
            if gap<=0: continue
            if gap>mg: break
            if A["cls"]!=B["cls"]: continue
            d=min(np.linalg.norm(A["p1"]+A["v1"]*gap-B["p0"]),
                  np.linalg.norm(B["p0"]-B["v0"]*gap-A["p1"]))
            gate=max(gate_mult*max(A["size"],B["size"]),min_gate)*(1+gap/mg)
            if d<gate and d<bd: bd,best=d,b
        if best is not None: used.add(best); pr+=1; par[find(best)]=find(a)
    df=df.copy(); df["track_id"]=df.track_id.map(lambda t: find(t) if t in par else t)
    return df,pr
def reinterp(df,stride):
    out=[]
    for tid,g in df.groupby("track_id"):
        g=g.sort_values("frame_idx").drop_duplicates("frame_idx").set_index("frame_idx")
        g=g.reindex(range(int(g.index.min()),int(g.index.max())+1,stride))
        g["interpolated"]=g["cx"].isna()
        c=["x1","y1","x2","y2","cx","cy_foot","w","h","conf"]
        g[c]=g[c].interpolate("linear"); g["track_id"]=tid
        for k in ["class_id","class_name","class_group"]: g[k]=g[k].ffill().bfill()
        out.append(g.reset_index().rename(columns={"index":"frame_idx"}))
    return pd.concat(out,ignore_index=True)
if __name__=="__main__":
    src,dst=sys.argv[1],sys.argv[2]; st=int(sys.argv[3]) if len(sys.argv)>3 else 3
    df=pd.read_parquet(src); print(f"before {df.track_id.nunique()} tracks")
    df,pr=stitch(df); print(f"merged {pr} pairs")
    df=reinterp(df,st); df["time_sec"]=df.frame_idx/29.97
    df.to_parquet(dst,index=False)
    L=df.groupby("track_id").size()
    print(f"after {df.track_id.nunique()} tracks")
    print(f"len mean {L.mean():.1f} median {L.median():.0f} p90 {L.quantile(.9):.0f}")
    print(f"objects/frame mean {df.groupby('frame_idx').size().mean():.1f}")
    print(f"interpolated {100*df.interpolated.mean():.1f}%")
    print(df.groupby('track_id').class_group.first().value_counts())

Writing stitch.py


In [5]:
%%writefile render.py
import sys, cv2, numpy as np, pandas as pd
C={"car":(0,200,255),"LGV":(0,255,120),"HGV":(255,80,0),"bus":(255,0,200),
   "motorcycle":(80,80,255),"cyclist":(255,255,0),"pedestrian":(0,255,255),
   "three-wheeler":(180,120,255)}
pq,vid,out=sys.argv[1],sys.argv[2],sys.argv[3]
t0=float(sys.argv[4]) if len(sys.argv)>4 else 0
dur=float(sys.argv[5]) if len(sys.argv)>5 else 90
df=pd.read_parquet(pq)
st=int(np.median(np.diff(sorted(df.frame_idx.unique())))) or 3
cap=cv2.VideoCapture(vid); assert cap.isOpened()
fps=cap.get(cv2.CAP_PROP_FPS) or 29.97
W=int(cap.get(3)); H=int(cap.get(4)); OW=1920; OH=int(1920*H/W); sx,sy=OW/W,OH/H
f0=max(int(t0*fps),int(df.frame_idx.min())); f1=min(int((t0+dur)*fps),int(df.frame_idx.max()))
d=df[(df.frame_idx>=f0)&(df.frame_idx<=f1)]
by={k:v for k,v in d.groupby("frame_idx")}
vw=cv2.VideoWriter(out,cv2.VideoWriter_fourcc(*"mp4v"),fps/st,(OW,OH))
cap.set(cv2.CAP_PROP_POS_FRAMES,f0); i=f0; n=0
while i<=f1:
    if not cap.grab(): break
    if i in by:
        ok,fr=cap.retrieve()
        if ok:
            fr=cv2.resize(fr,(OW,OH))
            for r in by[i].itertuples():
                c=C.get(r.class_group,(255,255,255))
                p1=(int(r.x1*sx),int(r.y1*sy)); p2=(int(r.x2*sx),int(r.y2*sy))
                cv2.rectangle(fr,p1,p2,c,2)
                cv2.putText(fr,str(int(r.track_id)),(p1[0],max(p1[1]-4,10)),
                            cv2.FONT_HERSHEY_SIMPLEX,0.45,c,1,cv2.LINE_AA)
            cv2.putText(fr,f"t={i/fps:6.1f}s  n={len(by[i])}",(12,28),
                        cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2,cv2.LINE_AA)
            vw.write(fr); n+=1
    i+=1
cap.release(); vw.release()
print(f"wrote {out}: {n} frames, {n*st/fps:.1f}s")

Writing render.py


In [6]:
!python track.py --video "$VID" --out-dir /content/clip90 \
  --start-sec 60 --max-seconds 90 --stride 3 --device cuda

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

best.pt: downloading bytes:  44% 8.39M/19.2M [00:00<00:00, 11.1MB/s]
best.pt: downloading bytes: 100% 19.1M/19.1M [00:00<00:00, 22.8MB/s, 1.87MB/s  ]
best.pt: reconstructing file: 100% 19.2M/19.2M [00:00<00:00, 22.9MB/s, 1.89MB/s  ]
3840x2160 @29.97 frames 1798->4495 sampled 899
[tracker] {}
/content/track.py:22: FutureWarning: The `ByteTrack` was deprecated since v0.28.0. It will be removed in v0.31.0.
  return sv.ByteTrack(**kw)
50/899 1.00 f/s ETA 14.1 min
100/899 1.28 f/s ETA 10.4 min
150/899 1.42 f/s ETA 8.8 min
200/899 1.51 f/s ETA 7.7 min
250/899 1.55 f/s ETA 7.0 min
300/899 1.58 f/s ETA 6.3 min
350/899 1.60 f/s ETA 5.7 min
400/899 1.61 f/s ETA 5.2 min
450/899 1.61 f/s ETA 

In [7]:
!python stitch.py /content/clip90/tracks.parquet /content/clip90/tracks_final.parquet 3
!python render.py /content/clip90/tracks_final.parquet "$VID" /content/clip.mp4 60 90
!ffmpeg -y -loglevel error -i /content/clip.mp4 -vcodec libx264 -pix_fmt yuv420p -crf 23 /content/clip_h264.mp4
!mkdir -p /content/drive/MyDrive/flytbase_out
!cp /content/clip90/tracks_final.parquet /content/clip_h264.mp4 /content/drive/MyDrive/flytbase_out/
!ffmpeg -y -loglevel error -i "$VID" -map 0:s:0 /content/drive/MyDrive/flytbase_out/int.srt
!ls -la /content/drive/MyDrive/flytbase_out/

before 1929 tracks
merged 741 pairs
after 1188 tracks
len mean 162.2 median 70 p90 421
objects/frame mean 214.3
interpolated 14.0%
class_group
pedestrian       471
motorcycle       433
car              184
LGV               33
cyclist           25
bus               19
HGV               17
three-wheeler      6
Name: count, dtype: int64
wrote /content/clip.mp4: 899 frames, 90.0s
total 122801
-rw------- 1 root root 117288454 Aug 22 08:21 clip_h264.mp4
-rw------- 1 root root   3949084 Aug 22 08:22 int.srt
-rw------- 1 root root   4509491 Aug 22 08:21 tracks_final.parquet
